# GEO Maps Python 

In [18]:
import plotly.express as px

df = px.data.election ()
geojson = px.data.election_geojson()

print (df['district'][2])
print (geojson['features'][0]['properties'])

11-Sault-au-Récollet
{'district': '11-Sault-au-Récollet'}


In [19]:
df.head()

,district,Coderre,Bergeron,Joly,total,winner,result,district_id
0,101-Bois-de-Liesse,2481,1829,3024,7334,Joly,plurality,101
1,102-Cap-Saint-Jacques,2525,1163,2675,6363,Joly,plurality,102
2,11-Sault-au-Récollet,3348,2770,2532,8650,Coderre,plurality,11
3,111-Mile-End,1734,4782,2514,9030,Bergeron,majority,111
4,112-DeLorimier,1770,5933,3044,10747,Bergeron,majority,112


In [20]:
fig = px.choropleth_mapbox(df, geojson=geojson, color="Bergeron",
                           locations="district", featureidkey="properties.district",
                           center={"lat": 45.5517, "lon": -73.7073},
                           mapbox_style="carto-positron", zoom=9)

fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

C:\Users\dell\AppData\Local\Temp\ipykernel_16936\3780487446.py:1: DeprecationWarning: *choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.choropleth_mapbox(df, geojson=geojson, color="Bergeron",


In [21]:
import pandas as pd

url = "population.csv"
df = pd.read_csv(url)

print(df.columns)
print(df.head())

Index(['Entity', 'Code', 'Year', 'Population'], dtype='str')
        Entity Code   Year  Population
0  Afghanistan  AFG -10000       14737
1  Afghanistan  AFG  -9000       20405
2  Afghanistan  AFG  -8000       28253
3  Afghanistan  AFG  -7000       39120
4  Afghanistan  AFG  -6000       54166


In [22]:
import pandas as pd
import plotly.express as px

url = "population.csv"
df = pd.read_csv(url)

# --- detect column names ---
# year column might be "year" or "Year"
year_col = "year" if "year" in df.columns else ("Year" if "Year" in df.columns else None)

if year_col is None:
    # Fallback: if there is no explicit year column, OWID file may already be single-year data
    # In that case, we will map using whatever population column exists.
    print("No explicit year column found. Columns:", df.columns)
    # choose population-like column
    pop_col = "population" if "population" in df.columns else df.columns[-1]
    dff = df.copy()
else:
    # population column is usually called "population"
    pop_col = "population" if "population" in df.columns else df.columns[-1]

    # choose the year to map
    year = 2022
    dff = df[df[year_col] == year].copy()

# detect country name column
country_col = "Entity" if "Entity" in dff.columns else ("Country" if "Country" in dff.columns else dff.columns[0])

fig = px.choropleth(
    dff,
    locations=country_col,
    locationmode="country names",
    color=pop_col,
    hover_name=country_col,
    color_continuous_scale="Viridis",
    title=f"World Population ({year if year_col is not None else 'all available'})"
)

fig.update_layout(
    geo=dict(showframe=False, showcoastlines=True, projection_type="natural earth"),
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.show()

C:\Users\dell\AppData\Local\Temp\ipykernel_16936\1722851208.py:29: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


## Lets Make a Covid 19 Map 

In [1]:
import pandas as pd
import plotly.express as px

In [9]:


# Set Max column print option 


pd.set_option('display.max_columns', None)

df = pd.read_csv('COVID_data.csv')


In [10]:
df.head()

,Date_reported,Country_code,Country,WHO_region,New_cases,Cumulative_cases,New_deaths,Cumulative_deaths
0,2020-01-03,AF,Afghanistan,EMRO,0,0,0,0
1,2020-01-04,AF,Afghanistan,EMRO,0,0,0,0
2,2020-01-05,AF,Afghanistan,EMRO,0,0,0,0
3,2020-01-06,AF,Afghanistan,EMRO,0,0,0,0
4,2020-01-07,AF,Afghanistan,EMRO,0,0,0,0


In [11]:
df['New_deaths'].value_counts()

New_deaths
 0        227193
 1         15129
 2          8687
 3          6273
 4          4876
 5          3944
 6          3350
 7          2849
 8          2605
 9          2264
 10         2058
 11         1757
 12         1688
 13         1355
 14         1354
 15         1257
 16         1094
 17         1082
 18          950
 19          903
 20          810
 21          755
 22          670
 23          625
 24          617
 26          583
 25          559
 27          536
 30          528
 28          508
 29          506
 32          451
 31          442
 34          434
 37          393
 33          389
 35          388
 36          373
 38          363
 39          362
 41          359
 42          341
 40          329
 44          309
 43          301
 45          295
 48          293
 46          289
 49          275
 47          265
 51          249
 53          248
 50          246
 52          239
 54          228
 57          216
 59          202
 60          200
 61

In [14]:
# Let's take an average of total cases by country location using group by function and create a new df 
df_total_cases = df.groupby(['Country'])['New_cases'].mean().reset_index()

df_total_cases

,Country,New_cases
0,Afghanistan,165.676686
1,Albania,244.934018
2,Algeria,199.304985
3,American Samoa,6.115103
4,Andorra,35.201613
5,Angola,77.683284
6,Anguilla,2.862170
7,Antigua and Barbuda,6.675953
8,Argentina,7371.390029
9,Armenia,329.565982


In [24]:
# Create the map visualization
fig = px.choropleth(
    df_total_cases,
    locations='Country',
    locationmode='country names',
    color='New_cases',
    title='Average New Cases by Country',
    range_color=(0, 5000000)
)

fig.update_coloraxes(cmin=0, cmax=5000000)

# Display the map
fig.show()

C:\Users\dell\AppData\Local\Temp\ipykernel_6372\3240698831.py:2: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


In [28]:
# Create the map visualization
fig = px.choropleth(
    df_total_cases,
    locations='Country',
    locationmode='country names',
    color='New_cases',
    title='Average New Cases by Country',
    range_color=(0, 5000000),
    animation_frame= "Country"
)

fig.update_coloraxes(cmin=0, cmax=5000000)

# Display the map
fig.show()

C:\Users\dell\AppData\Local\Temp\ipykernel_6372\1539916663.py:2: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


## Animate this graph using Monthly Data, Convert the Data into monthly Data and then animated the plot save it as avi mp4 file or gif 